# Deep Learning Grid Search – Pima Indians Diabetes

**Objective:** Use Scikit-learn GridSearchCV with a Keras deep learning model to tune:
- Batch size and training epochs
- Learning rate
- Network weight initialization
- Activation function
- Dropout regularization
- Number of neurons in the hidden layer


## Task 1: Data Acquisition and Loading

In [ ]:
# Install required packages if needed
# Uncomment the next line in Google Colab/Jupyter if TensorFlow or scikeras is missing
# %pip install -q scikeras tensorflow

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scikeras.wrappers import KerasClassifier

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Pima Indians Diabetes dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(url, names=columns)

print("Dataset shape:", df.shape)
df.head()


## Task 2: Data Preprocessing

In [ ]:
# Check missing values and basic information
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nClass distribution:")
print(df["Outcome"].value_counts())


In [ ]:
# In the Pima dataset, zero is not physiologically valid for several medical measurements.
# Replace zeros with NaN and impute them using the median.

zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

# Median imputation
df[zero_as_missing] = df[zero_as_missing].fillna(df[zero_as_missing].median())

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


## Task 3: Deep Learning Grid Search

A Keras model is wrapped using **SciKeras**, allowing it to work with Scikit-learn's `GridSearchCV`.

To keep the grid practical for a lab assignment, the search below tunes all requested parameters while using a modest number of combinations.


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.initializers import GlorotUniform, HeUniform

def create_model(learning_rate=0.001,
                 init_mode="glorot_uniform",
                 activation="relu",
                 dropout_rate=0.0,
                 neurons=16):
    model = Sequential([
        Dense(
            neurons,
            input_shape=(X_train.shape[1],),
            kernel_initializer=init_mode,
            activation=activation
        ),
        Dropout(dropout_rate),
        Dense(
            1,
            kernel_initializer=init_mode,
            activation="sigmoid"
        )
    ])

    optimizer = Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

classifier = KerasClassifier(
    model=create_model,
    verbose=0,
    random_state=42
)

# Standardization is performed inside the sklearn pipeline.
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", classifier)
])

param_grid = {
    "classifier__batch_size": [16, 32],
    "classifier__epochs": [20, 40],
    "classifier__model__learning_rate": [0.001, 0.01],
    "classifier__model__init_mode": ["glorot_uniform", "he_uniform"],
    "classifier__model__activation": ["relu", "tanh"],
    "classifier__model__dropout_rate": [0.0, 0.2],
    "classifier__model__neurons": [16, 32]
}

print("Grid parameters:")
for key, values in param_grid.items():
    print(f"{key}: {values}")


In [ ]:
# Run Grid Search
# Total combinations = 2^7 = 128; with cv=3 this trains 384 models.
# This can take several minutes depending on your CPU/GPU.

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

grid_result = grid.fit(X_train, y_train)

print("\nBest CV Accuracy:", grid_result.best_score_)
print("\nBest Parameters:")
for key, value in grid_result.best_params_.items():
    print(f"{key}: {value}")


In [ ]:
# Evaluate the best model on the unseen test set
best_model = grid_result.best_estimator_

y_pred = best_model.predict(X_test).ravel()
y_pred = (y_pred >= 0.5).astype(int)

test_accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", test_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# View the top GridSearchCV results
results = pd.DataFrame(grid_result.cv_results_)

top_results = results[
    ["mean_test_score", "std_test_score", "rank_test_score", "params"]
].sort_values("rank_test_score").head(10)

top_results


## Conclusion

GridSearchCV was successfully applied to a Keras deep learning model through SciKeras. The search tuned:

1. **Batch size**
2. **Number of epochs**
3. **Learning rate**
4. **Weight initialization**
5. **Activation function**
6. **Dropout regularization**
7. **Number of neurons in the hidden layer**

The combination producing the highest cross-validation accuracy is reported as the **best parameter set**. The selected model is then evaluated on the held-out test set.
